# Functional tests

Functional tests start the real application and use it through its web API, exactly as the website
does. Only fictional documents are sent: the ICAO specimen passport and images generated on the fly.

The checks live in [`scripts/functional_tests.py`](../scripts/functional_tests.py). This notebook
starts the app on a free local port, runs them, and stops the app again. It takes about 2 minutes.

| Group | What is checked |
|---|---|
| Normal use | Health, home page, session, a full scan with correct fields, dates and MRZ, the preview |
| Different inputs | Documents rotated 90°, 180° and 270°; a single-page PDF; front and back together |
| Bad input | No session, wrong file type, oversized file, invalid options, tiny image, blank page |
| Privacy and control | Scan deleted once delivered; cancelling; another user cannot see your scan |

That nothing is ever written to disk is checked separately by a backend unit test, which fails if
an upload could spill into a temporary file.

In [1]:
from report_helpers import *  # noqa: F403  (paths and small display helpers)

In [2]:
sys.path.insert(0, str(ROOT / "scripts"))
import functional_tests

with running_server() as base:
    results = functional_tests.run(base)

table(["#", "Check", "Result"],
      [[number, name, "✅" if ok else "❌ " + detail] for number, (name, ok, detail) in enumerate(results, 1)])
print(f"{sum(ok for _, ok, _ in results)} of {len(results)} functional checks passed")

| # | Check | Result |
|---|---|---|
| 1 | Health check reports the OCR engine ready | ✅ |
| 2 | Website home page loads | ✅ |
| 3 | Scanning without a session is refused (401) | ✅ |
| 4 | A browser session can be created | ✅ |
| 5 | Specimen passport scans successfully | ✅ |
| 6 | Name and passport number are correct | ✅ |
| 7 | Dates are correct | ✅ |
| 8 | MRZ check digits pass | ✅ |
| 9 | Corrected preview arrives with the result | ✅ |
| 10 | Scan is deleted as soon as its result is delivered | ✅ |
| 11 | Document rotated 90 degrees is read correctly | ✅ |
| 12 | Document rotated 180 degrees is read correctly | ✅ |
| 13 | Document rotated 270 degrees is read correctly | ✅ |
| 14 | Single-page PDF is read correctly | ✅ |
| 15 | Front and back can be uploaded together | ✅ |
| 16 | A non-image file is refused (415) | ✅ |
| 17 | An oversized file is refused (413) | ✅ |
| 18 | Invalid scan options are refused (422) | ✅ |
| 19 | A too-small image fails with a clear message | ✅ |
| 20 | A blank page completes with a warning instead of crashing | ✅ |
| 21 | One-shot /extract endpoint returns the fields | ✅ |
| 22 | A running scan can be cancelled | ✅ |
| 23 | Another user cannot see someone else's scan (404) | ✅ |

23 of 23 functional checks passed


## Run from a terminal instead

```powershell
.\start.ps1                                                        # terminal 1
.\.venv\Scripts\python.exe scripts\functional_tests.py --base http://127.0.0.1:8000   # terminal 2
```